# Exercise 4

Blocking simulation one. Main thing is comparing the different arrival and service assumptions.

## helpers

Need the event sim stuff first. Not very exciting but yeah.

In [ ]:
import heapq
import math
import os
import random
import statistics

try:
    import numpy as np
except ImportError:
    np = None

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    plt = None


PLOT_DIR = "pics"


def erlang_b(m, A):
    top = A**m / math.factorial(m)
    bottom = sum(A**i / math.factorial(i) for i in range(m + 1))
    return top / bottom


def ci_95(values):
    # t critical value for 95 percent CI with 10 replications, df = 9
    t_value = 2.262
    if np is not None:
        mean_value = float(np.mean(values))
        sd_value = float(np.std(values, ddof=1))
    else:
        mean_value = statistics.mean(values)
        sd_value = statistics.stdev(values)
    half_width = t_value * sd_value / math.sqrt(len(values))
    return mean_value, mean_value - half_width, mean_value + half_width


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def exp_mean(mean):
    return random.expovariate(1.0 / mean)


def erlang_arrival(mean=1.0, k=2):
    # Erlang(k, rate), with mean k/rate = mean
    rate = k / mean
    total = 0.0
    for _ in range(k):
        total += random.expovariate(rate)
    return total


def hyperexp_arrival():
    # mixture from the exercise text
    if random.random() < 0.8:
        return random.expovariate(0.8333)
    return random.expovariate(5.0)


def pareto_service(mean=8.0, k=2.05):
    xm = mean * (k - 1.0) / k
    u = random.random()
    return xm / (u ** (1.0 / k))


def simulate(m, n_arrivals, next_interarrival, next_service):
    clock = 0.0
    busy = 0
    blocked = 0
    departures = []

    for _ in range(n_arrivals):
        clock += next_interarrival()

        while departures and departures[0] <= clock:
            heapq.heappop(departures)
            busy -= 1

        if busy < m:
            busy += 1
            heapq.heappush(departures, clock + next_service())
        else:
            blocked += 1

    return blocked / n_arrivals


def run_case(case):
    reps = []
    for _ in range(10):
        reps.append(
            simulate(
                m=10,
                n_arrivals=10_000,
                next_interarrival=case["arrival_fn"],
                next_service=case["service_fn"],
            )
        )

    mean, low, high = ci_95(reps)
    return {
        "case": case["case"],
        "arrival": case["arrival"],
        "service": case["service"],
        "notes": case["notes"],
        "reps": reps,
        "mean": mean,
        "low": low,
        "high": high,
    }


def print_case(result):
    print(result["case"])
    print("-" * len(result["case"]))
    print("replications:", " ".join(f"{x:.5f}" for x in result["reps"]))
    print(f"mean blocked fraction: {result['mean']:.6f}")
    print(f"95% CI: [{result['low']:.6f}, {result['high']:.6f}]")
    print()


def print_table(results):
    headers = [
        "Case",
        "Arrival distribution",
        "Service distribution",
        "Mean blocked fraction",
        "95% confidence interval",
        "Notes",
    ]

    rows = []
    for r in results:
        rows.append(
            [
                r["case"],
                r["arrival"],
                r["service"],
                f"{r['mean']:.6f}",
                f"[{r['low']:.6f}, {r['high']:.6f}]",
                r["notes"],
            ]
        )

    widths = []
    for i, header in enumerate(headers):
        widths.append(max(len(header), max(len(row[i]) for row in rows)))

    print("Final comparison table")
    print("----------------------")
    print(" | ".join(headers[i].ljust(widths[i]) for i in range(len(headers))))
    print("-+-".join("-" * w for w in widths))
    for row in rows:
        print(" | ".join(row[i].ljust(widths[i]) for i in range(len(row))))


def save_blocking_plot(results):
    if plt is None:
        print("matplotlib not available, skipping plot.")
        return

    ensure_dir(PLOT_DIR)

    labels = [r["case"] for r in results]
    means = [r["mean"] for r in results]
    lower_errors = [r["mean"] - r["low"] for r in results]
    upper_errors = [r["high"] - r["mean"] for r in results]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(labels, means, color="steelblue", alpha=0.85)
    ax.errorbar(
        labels,
        means,
        yerr=[lower_errors, upper_errors],
        fmt="none",
        ecolor="black",
        capsize=4,
    )
    ax.set_ylabel("Blocked fraction")
    ax.set_title("Exercise 4: blocking probability by case")
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    out_path = os.path.join(PLOT_DIR, "ex4_blocking_comparison.png")
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    print(f"Saved plot to {out_path}")




## run

Here I do the cases from the exercise and then compare the blocked fractions.

In [ ]:
random.seed(12345)

m = 10
mean_interarrival = 1.0
mean_service = 8.0
A = mean_service / mean_interarrival
exact = erlang_b(m, A)

print("Exact Erlang-B check")
print("m =", m)
print("A =", A)
print("exact =", round(exact, 6))
print()

cases = [
    {
        "case": "Part 1",
        "arrival": "Poisson / exponential interarrival, mean 1",
        "service": "Exponential, mean 8",
        "notes": f"baseline, exact = {exact:.6f}",
        "arrival_fn": lambda: exp_mean(1.0),
        "service_fn": lambda: exp_mean(8.0),
    },
    {
        "case": "Part 2a",
        "arrival": "Erlang interarrival, k = 2, mean 1",
        "service": "Exponential, mean 8",
        "notes": "less variable arrivals",
        "arrival_fn": lambda: erlang_arrival(mean=1.0, k=2),
        "service_fn": lambda: exp_mean(8.0),
    },
    {
        "case": "Part 2b",
        "arrival": "Hyperexponential, p = 0.8/0.2",
        "service": "Exponential, mean 8",
        "notes": "more bursty arrivals",
        "arrival_fn": hyperexp_arrival,
        "service_fn": lambda: exp_mean(8.0),
    },
    {
        "case": "Part 3a",
        "arrival": "Poisson / exponential interarrival, mean 1",
        "service": "Constant, S = 8",
        "notes": "same mean service time",
        "arrival_fn": lambda: exp_mean(1.0),
        "service_fn": lambda: 8.0,
    },
    {
        "case": "Part 3b",
        "arrival": "Poisson / exponential interarrival, mean 1",
        "service": "Pareto Type I, k = 1.05, mean 8",
        "notes": "very heavy tail",
        "arrival_fn": lambda: exp_mean(1.0),
        "service_fn": lambda: pareto_service(mean=8.0, k=1.05),
    },
    {
        "case": "Part 3c",
        "arrival": "Poisson / exponential interarrival, mean 1",
        "service": "Pareto Type I, k = 2.05, mean 8",
        "notes": "still heavy but not as bad",
        "arrival_fn": lambda: exp_mean(1.0),
        "service_fn": lambda: pareto_service(mean=8.0, k=2.05),
    },
    {
        "case": "Part 3d",
        "arrival": "Poisson / exponential interarrival, mean 1",
        "service": "Uniform(0, 16)",
        "notes": "extra one",
        "arrival_fn": lambda: exp_mean(1.0),
        "service_fn": lambda: random.uniform(0.0, 16.0),
    },
    {
        "case": "Part 3e",
        "arrival": "Poisson / exponential interarrival, mean 1",
        "service": "Gamma, shape 2, scale 4",
        "notes": "extra one too",
        "arrival_fn": lambda: exp_mean(1.0),
        "service_fn": lambda: random.gammavariate(2.0, 4.0),
    },
]

results = []
for case in cases:
    result = run_case(case)
    results.append(result)
    print_case(result)

print_table(results)
save_blocking_plot(results)

print()
print("looks fine i think")
